# The Euphoric Void — Free Cloud Brain (Google Colab T4 GPU)
Runs **Llama 3.1 Nemotron 8B UltraLong 4M Instruct (Q4_K_S, 8,192 ctx)** on Google's free cloud GPU and prints an HTTPS tunnel URL for the game.

1. Click **Runtime → Change runtime type → T4 GPU → Save**.
2. Click the **Play** button on the cell below (takes ~2 minutes the first time).
3. Copy the **`https://...trycloudflare.com/v1`** URL it prints and paste it into **AI Brain → Colab / LM Studio URL** in the game.

In [ ]:
import os, re, subprocess, time, urllib.request

# 1. Install zstd (required by Ollama installer), Ollama, and Cloudflare Tunnel
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. Start Ollama with CORS & external host enabled
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)

# 3. Pull Llama 3.1 Nemotron 8B UltraLong 4M Instruct (Q4_K_S) & configure 8,192 context
!ollama pull hf.co/bartowski/nvidia_Llama-3.1-8B-UltraLong-4M-Instruct-GGUF:Q4_K_S
with open("Modelfile", "w") as f:
    f.write("FROM hf.co/bartowski/nvidia_Llama-3.1-8B-UltraLong-4M-Instruct-GGUF:Q4_K_S\nPARAMETER num_ctx 8192\n")
!ollama create nemotron-8b-ultralong -f Modelfile

# 4. Open public HTTPS tunnel and print your game URL
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nREADY! Paste this URL into AI Brain -> Colab / LM Studio URL:")
        print(f"{m.group(0)}/v1")
        break

# Keep tunnel alive while you play
proc.wait()
